In [20]:
import pandas as pd
import numpy as np


train_behaviors_path = "./data/train/behaviors.parquet"
train_articles_path = "./data/train/articles.parquet"
train_history_path = "./data/train/history.parquet"

validation_behaviors_path = "./data/validation/behaviors.parquet"
validation_articles_path = "./data/validation/articles.parquet"
validation_history_path = "./data/validation/history.parquet"

In [21]:

df_train_behaviors_prob = pd.read_parquet(train_behaviors_path).head(5)
print("="*25 + " behaviors列名及数据类型 " + "="*25)
print(df_train_behaviors_prob.dtypes)


========================= behaviors列名及数据类型 =========================
impression_id                     uint32
article_id                       float64
impression_time           datetime64[us]
read_time                        float32
scroll_percentage                float32
device_type                         int8
article_ids_inview                object
article_ids_clicked               object
user_id                           uint32
is_sso_user                         bool
gender                           float64
postcode                         float64
age                              float64
is_subscriber                       bool
session_id                        uint32
next_read_time                   float32
next_scroll_percentage           float32
dtype: object


In [22]:
display(df_train_behaviors_prob[['user_id', 'impression_time', 'article_ids_inview', 'article_ids_clicked']].head(2))

,user_id,impression_time,article_ids_inview,article_ids_clicked
0,139836,2023-05-24 07:47:53,"[9778623, 9778682, 9778669, 9778657, 9778736, ...",[9778657]
1,143471,2023-05-24 07:33:25,"[9778718, 9778728, 9778745, 9778669, 9778657, ...",[9778623]


In [23]:
df_train_articles_prob = pd.read_parquet(train_articles_path).head(5)
print("="*25 + " articles列名及数据类型 " + "="*25)
print(df_train_articles_prob.dtypes)

========================= articles列名及数据类型 =========================
article_id                     int32
title                         object
subtitle                      object
last_modified_time    datetime64[us]
premium                         bool
body                          object
published_time        datetime64[us]
image_ids                     object
article_type                  object
url                           object
ner_clusters                  object
entity_groups                 object
topics                        object
category                       int16
subcategory                   object
category_str                  object
total_inviews                float64
total_pageviews              float64
total_read_time              float32
sentiment_score              float32
sentiment_label               object
dtype: object


In [24]:
display(df_train_articles_prob[['article_id', 'category', 'subcategory', 'sentiment_label', 'published_time']].head(2))

,article_id,category,subcategory,sentiment_label,published_time
0,3001353,140,[],Negative,2006-08-31 08:06:45
1,3003065,414,"[433, 434]",Positive,2006-05-21 16:57:00


In [25]:
df_train_history_prob = pd.read_parquet(train_history_path).head(5)
print("="*25 + " history列名及数据类型 " + "="*25)
print(df_train_history_prob.dtypes)

========================= history列名及数据类型 =========================
user_id                    uint32
impression_time_fixed      object
scroll_percentage_fixed    object
article_id_fixed           object
read_time_fixed            object
dtype: object


In [26]:
display(df_train_history_prob[['user_id', 'article_id_fixed', 'impression_time_fixed']].head(2))

,user_id,article_id_fixed,impression_time_fixed
0,13538,"[9738663, 9738569, 9738663, 9738490, 9738663, ...","[2023-04-27T10:17:43.000000, 2023-04-27T10:18:..."
1,14241,"[9738557, 9738528, 9738533, 9738684, 9739035, ...","[2023-04-27T09:40:18.000000, 2023-04-27T09:40:..."


In [19]:
import torch
from torch.utils.data import Dataset
import polars as pl


def build_dataset_dict(df_history_path, df_articles_path):
    df_history = pl.read_parquet(df_history_path)
    df_articles = pl.read_parquet(df_articles_path).with_columns(pl.col('published_time').dt.timestamp("ms").alias('published_ts'))

    # 构建users-history字典
    user_history_dict = dict(zip(df_history['user_id'], df_history['article_id_fixed'].to_list()))
    
    # 构建articles-info字典
    article_dict = dict(zip(df_articles['article_id'], df_articles['published_ts'].to_list()))

    return user_history_dict, article_dict

def explode_user_behaviors(behavor_path):
    df_behaviors = pl.scan_parquet(behavor_path)

    df_behaviors = (
        df_behaviors
        .explode('article_ids_inview')
        .with_columns(pl.col('article_ids_inview').is_in(pl.col('article_ids_clicked')).cast(pl.Int8).alias('labels'))
        .select(['user_id', 'article_ids_inview', 'labels'])
        .collect()
    )

    return df_behaviors.to_dicts()

# def padding_history()

class EbnerdDataset(Dataset):
    def __init__(self, history_dict, article_dict, behaviors_dict, MAX_HISTORY_LEN=50):
        self.history_dict = history_dict
        self.article_dict = article_dict
        self.behaviors_dict = behaviors_dict
        self.MAX_HISTORY_LEN = MAX_HISTORY_LEN

    def __len__(self):
        return len(self.behaviors_dict)
    
    def __getitem__(self, idx):
        # user_id, target_article_id, history_article_ids, target_article_published_ts, label
        user_behavior = self.behaviors_dict[idx]
        user_id = user_behavior['user_id']
        target_article_id = user_behavior['article_ids_inview']
        target_published_ts = self.article_dict.get(target_article_id, 0)
        label = user_behavior['labels']

        history = self.history_dict.get(user_id, [])[-self.MAX_HISTORY_LEN:]
        history_with_padding = history + [0] * (self.MAX_HISTORY_LEN - len(history))

        return {
            'user_id': torch.tensor([user_id], dtype=torch.long),
            'target_article_id': torch.tensor([target_article_id],dtype=torch.long),
            'history_article_ids': torch.tensor(history_with_padding,dtype=torch.long),
            'target_published_ts': torch.tensor([target_published_ts],dtype=torch.long),
            'label': torch.tensor([label], dtype=torch.float)
        }





In [20]:
behavior_path = "./data/train/behaviors.parquet"
history_path = "./data/train/history.parquet"
article_path = "./data/train/articles.parquet"

user_history_dict, article_dict = build_dataset_dict(history_path, article_path)
behaviors_dict = explode_user_behaviors(behavior_path)

ebnerdDataset = EbnerdDataset(user_history_dict, article_dict, behaviors_dict)
sample = ebnerdDataset[0]
print(sample)


{'user_id': tensor([139836]), 'target_article_id': tensor([9778623]), 'history_article_ids': tensor([9745590, 9748574, 9748432, 9748080, 9750687, 9750802, 9750829, 9750793,
        9750726, 9748041, 4800276, 9757533, 9760252, 9760528, 9760272, 9760067,
        9765336, 9765185, 9765156,       0,       0,       0,       0,       0,
              0,       0,       0,       0,       0,       0,       0,       0,
              0,       0,       0,       0,       0,       0,       0,       0,
              0,       0,       0,       0,       0,       0,       0,       0,
              0,       0]), 'target_published_ts': tensor([1684908515000]), 'label': tensor([0.])}


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DINAttention(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        """
        :param embedding_dim: 输入的embedding维度
        :param hidden_dim: 注意力网络的隐藏层维度
        """
        super(DINAttention, self).__init__()
        self.attention_mlp = nn.Sequential(
            nn.Linear(embedding_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, querys, keys, mask=None):
        """
        :param querys: [batch_size, 1, embedding_dim] 候选文章的embedding
        :param keys: [batch_size, seq_len, embedding_dim] 历史文章的embedding
        :param mask: [batch_size, seq_len] 用于屏蔽padding位置
        """
        # [!note] 使用.expand()将query的维度扩充成与keys相同以便于拼接，.expand(a,b,c)表示把三个维度分别扩充成a、b、c，-1表示不变
        input = torch.cat([
            querys.expand(-1, keys.shape[1], -1), 
            keys, 
            querys - keys, 
            querys * keys
        ], dim=-1)

        # attention_score形状(batch, seq_len, 1)
        attention_score = self.attention_mlp(input)

        # 不能写成if mask:
        if mask is not None:
            attention_score[mask == False] = -1e6

        attention_weights = F.softmax(attention_score)
        return attention_weights, keys


In [30]:
a = torch.tensor([[[1,2,3]],[[4,5,6]]])
print(f"a: \n{a}")
print(a.shape)
b = torch.tensor([[[1,1,1], [1,1,1]],[[2,2,2], [2,2,2]]])
print(f"b: \n{b}")
print(b.shape)
print("===============")
c = a - b
d = a * b
print(f"c: \n{c}")
print(f"c.shape{c.shape}")
print(f"d: \n{d}")
print(f"d.shape{d.shape}")

concated = torch.cat([a, b, c, d], dim=-1)
print(f"concated: \n{concated}")
print(f"concated.shape: {concated.shape}")

a: 
tensor([[[1, 2, 3]],

        [[4, 5, 6]]])
torch.Size([2, 1, 3])
b: 
tensor([[[1, 1, 1],
         [1, 1, 1]],

        [[2, 2, 2],
         [2, 2, 2]]])
torch.Size([2, 2, 3])
c: 
tensor([[[0, 1, 2],
         [0, 1, 2]],

        [[2, 3, 4],
         [2, 3, 4]]])
c.shapetorch.Size([2, 2, 3])
d: 
tensor([[[ 1,  2,  3],
         [ 1,  2,  3]],

        [[ 8, 10, 12],
         [ 8, 10, 12]]])
d.shapetorch.Size([2, 2, 3])


RuntimeError: Sizes of tensors must match except in dimension 2. Expected size 1 but got size 2 for tensor number 1 in the list.